In [12]:
import os
import json
import re
from pathlib import Path
from typing import Dict, List, Set, Optional

from tqdm import tqdm

from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, Session, SamplerV2 as Sampler
from qiskit_ibm_runtime.runtime_job import (
    RuntimeJobMaxTimeoutError,
    RuntimeJobFailureError,
    RuntimeInvalidStateError,
)
from qiskit.qpy import load

In [8]:
service = QiskitRuntimeService()

In [9]:
service.backends()

[<IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_sherbrooke')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_kingston')>]

In [25]:
def simulate_and_store_z_expectations_real_qpu_json(
    qpy_folder: str,
    output_json_path: str,
    shots: int = 1024,
    per_circuit_timeout_s: float = 60.0,
    backend_name: str = "ibm_fez",
    service: Optional[QiskitRuntimeService] = None,
) -> None:
    """
    Same as before but:
      * Keeps AerSimulator for ideal Z.
      * Runs the circuit on real hardware via SamplerV2 in a session.
      * Stores per-circuit expectations in a summary JSON.
      * Also writes shot-level JSONs (counts + bitstrings + Z) separately for ideal and hardware runs.
    """

    def counts_to_z_expectation(counts: Dict[str, int], n_qubits: int) -> List[float]:
        total = sum(counts.values())
        if total == 0:
            return [0.0] * n_qubits
        exp = [0.0] * n_qubits
        for key, cnt in counts.items():
            if key.startswith(("0x", "0X")):
                val = int(key, 16)
            else:
                val = int(key, 2)
            bits = bin(val)[2:].zfill(n_qubits)[::-1]  # little-endian
            for i in range(n_qubits):
                z = 1 if bits[i] == "0" else -1
                exp[i] += z * cnt
        return [round(e / total, 6) for e in exp]

    def unique_key(preferred: str, fallback_base: str, taken: Set[str]) -> str:
        base = preferred.strip() if preferred and preferred.strip() else fallback_base
        if base not in taken:
            taken.add(base)
            return base
        k = 2
        while True:
            cand = f"{base}#{k}"
            if cand not in taken:
                taken.add(cand)
                return cand
            k += 1

    def has_measurements(qc) -> bool:
        return any(inst.name == "measure" for inst, _, _ in qc.data)

    def sanitize(name: str) -> str:
        return re.sub(r"[^A-Za-z0-9._-]", "_", name)

    if service is None:
        service = QiskitRuntimeService()  # assume auth is already configured externally

    # Get backend object (not string) to feed into Session. :contentReference[oaicite:1]{index=1}
    backends = service.backends(name=backend_name)
    if not backends:
        raise RuntimeError(f"No backend found with name '{backend_name}'")
    backend_obj = backends[0]  # take first match

    ideal_sim = AerSimulator()

    results_dict: Dict[str, Dict[str, List[float]]] = {}
    taken_names: Set[str] = set()

    output_base = Path(output_json_path)
    shot_dir = output_base.parent / (output_base.stem + "_shot_data")
    shot_dir.mkdir(parents=True, exist_ok=True)

    # Open a session correctly: pass backend object as positional arg. :contentReference[oaicite:2]{index=2}
    with Session(backend_obj) as session:
        sampler = Sampler(mode=session)  # SamplerV2 uses mode=session. :contentReference[oaicite:3]{index=3}

        for file in tqdm(sorted(os.listdir(qpy_folder))):
            if not file.endswith(".qpy"):
                continue
            file_path = os.path.join(qpy_folder, file)
            try:
                with open(file_path, "rb") as f:
                    circuits = load(f)
            except Exception as e:
                print(f"⚠️ Error loading {file}: {e}")
                continue

            for idx, qc in enumerate(circuits):
                n_qubits = qc.num_qubits
                circuit_key = unique_key(qc.name, f"{Path(file).stem}_{idx}", taken_names)
                safe_key = sanitize(circuit_key)

                # === Ideal (Aer) run ===
                qc_ideal = qc.copy()
                if not has_measurements(qc_ideal):
                    qc_ideal = qc_ideal.copy()
                    qc_ideal.measure_all()
                try:
                    tqc_ideal = transpile(qc_ideal, backend=ideal_sim)
                    job_ideal = ideal_sim.run(tqc_ideal, shots=1, memory=True)
                    ideal_result = job_ideal.result()
                    ideal_counts = ideal_result.get_counts()
                    z_ideal = counts_to_z_expectation(ideal_counts, n_qubits)
                    # per-shot bitstrings (Aer provides memory). :contentReference[oaicite:4]{index=4}
                    try:
                        ideal_bitstrings = ideal_result.get_memory()
                    except Exception:
                        # fallback to expanding counts
                        ideal_bitstrings = []
                        for b, c in ideal_counts.items():
                            ideal_bitstrings.extend([b] * c)
                    ideal_payload = {
                        "z": z_ideal,
                        "counts": ideal_counts,
                        "bitstrings": ideal_bitstrings,
                    }
                    with open(shot_dir / f"{safe_key}__ideal_shots.json", "w") as f_out:
                        json.dump(ideal_payload, f_out, indent=2)
                except Exception as e:
                    print(f"❌ Aer (ideal) failed for '{circuit_key}': {e}")
                    continue  # skip hardware if ideal fails

                # === Hardware run via SamplerV2 ===
                qc_hw = qc.copy()
                if not has_measurements(qc_hw):
                    qc_hw = qc_hw.copy()
                    qc_hw.measure_all()
                try:
                    # print(qc_hw)
                    tqc_hw = transpile(qc_hw, backend=backend_obj)
                    print("transpile done ...")
                except Exception as e:
                    print(f"❌ Transpile-to-hardware failed for '{circuit_key}': {e}")
                    continue

                try:
                    job_hw = sampler.run([tqc_hw], shots=shots)
                    pub_result = job_hw.result(timeout=per_circuit_timeout_s)[0]
                    # Combined counts: prefer join_data if available, else fallback to meas.get_counts(). :contentReference[oaicite:5]{index=5}
                    try:
                        hw_counts = pub_result.join_data().get_counts()
                    except Exception:
                        # fallback: try the default classical register
                        try:
                            hw_counts = pub_result.data.meas.get_counts()
                        except Exception:
                            # last resort: collect all register counts
                            hw_counts = {}
                            for name, reg in vars(pub_result.data).items():
                                if hasattr(reg, "get_counts"):
                                    try:
                                        hw_counts[name] = reg.get_counts()
                                    except Exception:
                                        pass
                            # flatten if needed (user can adapt)
                    if not hw_counts:
                        print(f"⚠️ No counts extracted for hardware run '{circuit_key}'; skipping.")
                        continue
                    z_hardware = counts_to_z_expectation(hw_counts, n_qubits)

                    # Per-shot bitstrings from SamplerV2: use get_strings() which preserves order. :contentReference[oaicite:6]{index=6}
                    try:
                        hw_bitstrings = pub_result.data.meas.get_strings()
                    except Exception:
                        # fallback: expand counts (loses order)
                        hw_bitstrings = []
                        if isinstance(hw_counts, dict):
                            for b, c in hw_counts.items():
                                hw_bitstrings.extend([b] * c)
                    hw_payload = {
                        "z": z_hardware,
                        "counts": hw_counts,
                        "bitstrings": hw_bitstrings,
                    }
                    with open(shot_dir / f"{safe_key}__hardware_shots.json", "w") as f_out:
                        json.dump(hw_payload, f_out, indent=2)
                except RuntimeJobMaxTimeoutError as e:
                    print(f"⏱️ Hardware timeout for '{circuit_key}': {e}")
                    continue
                except (RuntimeJobFailureError, RuntimeInvalidStateError) as e:
                    print(f"❌ Hardware job error for '{circuit_key}': {e}")
                    continue
                except Exception as e:
                    print(f"❌ Unexpected hardware error for '{circuit_key}': {e}")
                    continue

                # === Summary aggregate ===
                results_dict[circuit_key] = {
                    "z_ideal": z_ideal,
                    "z_hardware": z_hardware,
                }
            break
    # Write summary JSON
    try:
        with open(output_json_path, "w") as out_f:
            json.dump(results_dict, out_f, indent=2)
        print(f"\n✅ Aggregated results saved to {output_json_path}")
        print(f"📁 Shot-level files in {shot_dir}")
    except Exception as e:
        print(f"❌ Failed to write summary JSON: {e}")


In [27]:
simulate_and_store_z_expectations_real_qpu_json(
    qpy_folder="../../../../../quantum/ExecutionResults/StoredCircuits/",
    output_json_path="real_z_expectations.json",
    shots=1024,
    service=service
)

  0%|                                                                                                         | 0/7004 [00:00<?, ?it/s]/tmp/ipykernel_114869/1770131622.py:47: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  return any(inst.name == "measure" for inst, _, _ in qc.data)


transpile done ...


  0%|                                                                                                         | 0/7004 [01:02<?, ?it/s]

❌ Unexpected hardware error for '10-qubit Deutsch Jozsa malicious': 'Timed out waiting for job to complete after 60.0 secs.'

✅ Aggregated results saved to real_z_expectations.json
📁 Shot-level files in real_z_expectations_shot_data
